As bibliotecas importadas serão explicadas conforme o uso ao decorrer do código.

Ademais, assim como anteriormente, a base de dados foi carregada usando a biblioteca Pandas, em que foi necessário um tratamento prévio da mesma, como visto na análise exploratória de dados.

In [34]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import cross_validate, KFold, ShuffleSplit

from ISLP.models import sklearn_sm, ModelSpec as MS

Auto = pd.read_csv('Auto.csv')
Auto = Auto[Auto['horsepower']!='?'].copy()
Auto['horsepower'] = Auto['horsepower'].astype(int)

Aqui, a biblioteca sklearn_sm foi importada uma vez que a ideia é utilizar a validação cruzada do scikit, e a regressão linear da biblioteca statsmodels. Essa técnica de cruzamento de funções é chamada de "wrapper".

Resumindo, a regressão é feita através da biblioteca statsmodels, equanto o algoritmo de validação cruzada é feito via scikit-learn. Na função cross-validate() é inserido o "motor" da regressão, a base de dados separada em preditores "X" e saída "Y" e por fim é inserido na variável cv o número de grupos a serem separados dentro dessa base de dados.

A ideia aqui é usar a abordagem LOOCV, logo o número de separações inserido foi de 392 (que é o valor de Auto.shape[0]). Com isso temos 392 MSE's, um para cada iteração do LOOCV, e sabemos que o MSE final a ser considerado é a média de todos eles.

Portanto, obtemos enfim o MSE médio para a regressão linear usando a abordagem de validação cruzada LOOCV de 24,23.

In [35]:
hp_model = sklearn_sm(sm.OLS,MS(['horsepower']))

X = Auto.drop(columns='mpg')
Y = Auto['mpg']

cv_results = cross_validate(hp_model,X,Y,cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])

print(f'{cv_err:.2f}')

24.23


Assim como na abordagem conjunto de validação, foi feito uma automação para aplicar diferentes ordens polinomiais para regressão, a fim de verificar qual produz o menor MSE médio de validação.

Contudo, foi refeito o mesmo processo anterior, mas agora para ordens polinomiais de 1 a 5.

In [36]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,X,Y,cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
    print(f'LOOCV - Ordem polinomial: {d} ==> MSE médio: {cv_error[i]:.2f}')

LOOCV - Ordem polinomial: 1 ==> MSE médio: 24.23
LOOCV - Ordem polinomial: 2 ==> MSE médio: 19.25
LOOCV - Ordem polinomial: 3 ==> MSE médio: 19.33
LOOCV - Ordem polinomial: 4 ==> MSE médio: 19.42
LOOCV - Ordem polinomial: 5 ==> MSE médio: 19.03


Posteriormente, praticamente o mesmo mecanismo foi implementado, em que a ideia aqui é utilizar a abordagem do K-fold, usando 10 separações. Então, a única diferença é que para a variável "cv" da biblioteca scikit foi inserido a divisão fornecida pela própria bilioteca, usando a função "KFold".

In [37]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,shuffle=True,random_state=0)

for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,X,Y,cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
    print(f'K-fold - Ordem polinomial: {d} ==> MSE médio: {cv_error[i]:.2f}')

K-fold - Ordem polinomial: 1 ==> MSE médio: 24.21
K-fold - Ordem polinomial: 2 ==> MSE médio: 19.19
K-fold - Ordem polinomial: 3 ==> MSE médio: 19.28
K-fold - Ordem polinomial: 4 ==> MSE médio: 19.48
K-fold - Ordem polinomial: 5 ==> MSE médio: 19.14


OBS: Mesmo para uma base de dados pequena, o tempo de rodagem do código para o LOOCV foi de 5,8 segundos, enquanto para o K-fold foi de apenas 0,1 segundos, obtendo um MSE médio muito semelhante à abordagem LOOCV.

Neste código, o autor mostra uma forma alternativa e flexível para a separação da base de dados para validação cruzada. Para tal, o autor utiliza a função "ShuffleSplit" da biblioteca skikit, que permite especificar o número de divisões e o tamanho do teste.

In [38]:
validation = ShuffleSplit(n_splits=1,test_size=196,random_state=0)

results = cross_validate(hp_model,Auto.drop(columns='mpg'),Auto['mpg'],cv=validation)

print(results['test_score'])

[23.61661707]


Ainda utilizando a função "ShuffleSplit", o código a seguir implementa a abordagem do K-fold, mostrando que é possível averiguar a variabilidade da abordagem, embora não seja possível interpretar como um desvio padrão da média, uma vez que existe correlação entre as separações da abordagem, uma vez que certos dados se sobrepõe a outros.

In [39]:
validation = ShuffleSplit(n_splits=10,test_size=196,random_state=0)

results = cross_validate(hp_model,Auto.drop(columns='mpg'),Auto['mpg'],cv=validation)

print(results['test_score'].mean(),results['test_score'].std())

23.802232661034164 1.4218450941091847
